# Decision Tree Learning (DTL): Submission Kaggle

Notebook ini menghasilkan `submission.csv` dari data mentah dengan satu pohon keputusan (kelas `DTL`).

Pohon C4.5 (gain ratio, cost-complexity pruning) sudah dilatih oleh `src/kaggle/dtl_new_train.py` dan
dimuat dari `output/dtl_new/dtl_model.json`. Cara pelatihan dan referensinya ada di skrip tersebut.

| Model | Macro F1, CV 5-fold (seed 11, 22, 33, 2026) |
|---|---|
| DTL pada `notebook.ipynb` | 0,8658 |
| DTL pada notebook ini | 0,8677 |

Skor CV dihitung dengan mengulang seluruh proses pelatihan di dalam setiap fold, dengan ambang 0,4.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src" / "kaggle" / "dataset" / "train.csv").exists())
sys.path.insert(0, str(REPO / "src" / "algoritma"))
sys.path.insert(0, str(REPO / "src" / "kaggle"))

from dtl import DTL
from dtl_new_features import build_features, clean, load_data

DATA = REPO / "src" / "kaggle" / "dataset"
OUTPUT = REPO / "output" / "dtl_new"
THRESHOLD = 0.4

train_raw, test = load_data(DATA)
print("train:", train_raw.shape, " test:", test.shape)

train: (27373, 19)  test: (6844, 18)


## Data Cleaning dan Transformation

Duplikat persis pada train dibuang. Nilai kosong diisi median atau modus train, disertai penanda kosong.
Fitur tambahan: tanggal booking, lama menginap, dan rata-rata label booking "saudara" (satu rombongan).
Untuk baris train, rata-rata itu dihitung out-of-fold. Semua statistik berasal dari train, dan label test tidak dipakai.

In [2]:
train = clean(train_raw)
X_train, X_test = build_features(train, train["is_canceled"], test)
print("train:", len(train_raw), "->", len(train), "baris")
print("fitur:", X_train.shape[1], " sisa nilai kosong di test:", int(X_test.isna().sum().sum()))

train: 27373 -> 27309 baris
fitur: 46  sisa nilai kosong di test: 0


## Memuat Pohon

In [3]:
model = DTL.load(OUTPUT / "dtl_model.json")
assert list(X_test.columns) == model.feature_names_
print("daun:", model.n_leaves_, " kedalaman:", model.depth_)


def show(node, depth, prefix=""):
    p = node.class_weights[1] / node.class_weights.sum()
    if node.leaf or depth == 0:
        print(f"{prefix}{'daun' if node.leaf else '...'}  P(batal)={p:.2f}")
        return
    print(f"{prefix}{model.feature_names_[node.feature]} <= {node.threshold:.3f}")
    show(node.left, depth - 1, prefix + "|  ya: ")
    show(node.right, depth - 1, prefix + "|  tidak: ")


show(model.root_, 3)

daun: 301  kedalaman: 29
sib_te <= 0.471
|  ya: sib_te <= 0.273
|  ya: |  ya: daun  P(batal)=0.01
|  ya: |  tidak: days_before_arrival <= 148.992
|  ya: |  tidak: |  ya: ...  P(batal)=0.20
|  ya: |  tidak: |  tidak: ...  P(batal)=0.67
|  tidak: sib_te <= 0.665
|  tidak: |  ya: nightly_rate_idr <= 683490.954
|  tidak: |  ya: |  ya: daun  P(batal)=0.10
|  tidak: |  ya: |  tidak: ...  P(batal)=0.88
|  tidak: |  tidak: special_requests <= 2.500
|  tidak: |  tidak: |  ya: daun  P(batal)=1.00
|  tidak: |  tidak: |  tidak: ...  P(batal)=0.89


## Prediksi dan Submission

Booking diprediksi batal jika P(batal) minimal 0,4.

In [4]:
labels = (model.predict_proba(X_test)[:, 1] >= THRESHOLD).astype(int)
submission = pd.DataFrame({"id": test["id"], "is_canceled": labels})

sample = pd.read_csv(DATA / "sample_submission.csv", dtype={"id": str})
assert list(submission.columns) == list(sample.columns)
assert submission["id"].tolist() == sample["id"].tolist()

submission.to_csv(OUTPUT / "submission.csv", index=False)
print("baris:", len(submission), " rasio batal:", round(labels.mean(), 4))
submission.head()

baris: 6844  rasio batal: 0.2972


,id,is_canceled
0,5cb6f473,1
1,c06b4ab3,1
2,22bf3fc3,0
3,25408a48,0
4,621586b6,0
